# Silver Layer — CRM: Product Info

Transforms the bronze `workspace.bronze.prd_info` table into `workspace.silver.product_info` with meaningful, readable column names.

---

## Table of Contents — Transformations Applied

| # | Transformation | Description |
|---|---|---|
| 1 | **Column Renaming** | Renamed abbreviated columns to meaningful names (e.g., `prd_id` → `product_id`, `prd_nm` → `product_name`) |
| 2 | **Trim Whitespace** | Trimmed leading/trailing spaces from `product_name` and `product_line` |
| 3 | **Product Line Mapping** | Mapped codes to full words: `R` → `Road`, `S` → `Sport`, `M` → `Mountain`, `T` → `Touring`, `NULL` → `Unknown` |
| 4 | **NULL Cost Handling** | Replaced NULL `cost` values with `0` for safe downstream aggregations |
| 5 | **NULL End Date Handling** | Added `is_active` boolean column (`True` when `end_date` is NULL, `False` otherwise) |
| 6 | **Category Key Extraction** | Added `Category_key` column — first 5 characters of `product_key` with hyphens replaced by underscores (e.g., `CO-RF-FR-R92B-58` → `CO_RF`) |

---

## Column Mapping (Bronze → Silver)

| Bronze Column | Silver Column | Transformation Applied |
|---|---|---|
| `prd_id` | `product_id` | Renamed |
| `prd_key` | `product_key` | Renamed |
| `prd_key` | `Category_key` | New column: first 5 characters of `prd_key` via `substr(1, 5)`, hyphens replaced with underscores |
| `prd_nm` | `product_name` | Renamed + Trimmed |
| `prd_cost` | `cost` | Renamed + NULLs replaced with `0` |
| `prd_line` | `product_line` | Renamed + Trimmed + Mapped (`R`→`Road`, `S`→`Sport`, `M`→`Mountain`, `T`→`Touring`, `NULL`→`Unknown`) |
| `prd_start_dt` | `start_date` | Renamed |
| `prd_end_dt` | `end_date` | Renamed (NULL kept = still active) |
| — | `is_active` | New column: `True` if `end_date` is NULL, `False` otherwise |

---

## Implementation

All transformations are implemented in **Spark SQL**:
- `TRIM()` — remove leading/trailing whitespace
- `CASE WHEN` — conditional mapping for product_line and is_active
- `COALESCE()` — replace NULL cost with 0
- `SUBSTRING()` — extract first 5 characters of `prd_key` for `Category_key`
- `REGEXP_REPLACE()` — replace hyphens with underscores in `Category_key`
- `CREATE OR REPLACE TABLE ... AS SELECT` — write to silver layer

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.product_info AS
SELECT
    prd_id                              AS product_id,
    prd_key                             AS product_key,
    REGEXP_REPLACE(SUBSTRING(prd_key, 1, 5), '-', '_') AS Category_key,
    TRIM(prd_nm)                        AS product_name,
    COALESCE(prd_cost, 0)               AS cost,
    CASE TRIM(prd_line)
        WHEN 'R' THEN 'Road'
        WHEN 'S' THEN 'Sport'
        WHEN 'M' THEN 'Mountain'
        WHEN 'T' THEN 'Touring'
        ELSE 'Unknown'
    END                                 AS product_line,
    prd_start_dt                        AS start_date,
    prd_end_dt                          AS end_date,
    prd_end_dt IS NULL                  AS is_active
FROM workspace.bronze.prd_info

In [0]:
%sql
SELECT * FROM workspace.silver.product_info LIMIT 10